# Import packages

In [ ]:
import os, requests
from pprint import pprint
import pandas as pd

# Custom functions

In [ ]:
from tools.create_jaspar_filters import obtain_jaspar_dataset, obtain_taxonomy_id, obtain_all_cell_lines, \
    cell_lines_df_clearing

# Download original JASPAR dataset
JASPAR: https://jaspar.elixir.no/downloads/
* Go for
    * Latest release
    * `CORE PFMs` collection for experimentally validated TF binding motifs with high quality
    * `Vertebrates` as the taxonomic group
    * `Single batch file (txt)`
    * `non-redundant` removes duplicate motifs for the same TF
    * `MEME` format can be used natively with fimo

In [ ]:
jaspar_url = 'https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme.txt'

jaspar_save_dir = 'input/jaspar_dataset'
os.makedirs(jaspar_save_dir, exist_ok=True)

In [ ]:
file_path = os.path.join(jaspar_save_dir, jaspar_url.split('/')[-1].split('.')[0] + '_original.txt')

response = requests.get(jaspar_url)

with open(file_path, "wb") as f:
    f.write(response.content)

print("Downloaded to:", file_path)

# Filter JASPAR for specific cell types
Same logic can be applied to other diseases for filtering

In [ ]:
filter_save_dir = 'input/filters'
os.makedirs(filter_save_dir, exist_ok=True)

## Obtain all melanoma cell lines

### (Optional) Obtain all NCIt code releated to a search term
NCI EVS API fields exploration: https://evsexplore.semantics.cancer.gov/evsexplore/evsapi

In [ ]:
ncit_search_term = 'melanoma'  #Seaching "melanoma" will also return results like "cutaneous melanoma"
page_size = 500  #This is for pagerization that NCIt has. It's a protection such that not all results return in 1 call but that also means you need to iterate and pool all the returns together

In [ ]:
all_concepts = obtain_jaspar_dataset(ncit_search_term, page_size)

In [ ]:
pprint(all_concepts, indent=4)

In [ ]:
ncit_dict = {dict['code']: dict['name'] for dict in all_concepts}  #All the other fields don't matter
pprint(ncit_dict)

In [ ]:
ncit_df = pd.DataFrame(ncit_dict.items(), columns=['code', 'concept'])
ncit_df.to_csv(
    os.path.join(filter_save_dir, f'NCIt_concepts_{ncit_search_term}.csv'),
    index=False
)  #Save NCIt outputs for future references

### Obtain taxonomy code
This can be used later to filter cell lines by species

In [ ]:
species_search_term = 'Homo sapiens'

In [ ]:
tax_id = obtain_taxonomy_id(species_search_term)

### Obtain related cell lines  related to a search term
Cellosaurus API methods & exploration: https://api.cellosaurus.org/api-methods#/

In [ ]:
cell_line_search_term = 'melanoma'
rows_size = 1000

In [ ]:
all_cell_lines_data = obtain_all_cell_lines(cell_line_search_term, tax_id, rows_size)

In [ ]:
cell_line_df = cell_lines_df_clearing(all_cell_lines_data)
cell_line_df = cell_line_df[[
    'cell_line_id',
    'cell_line_name',
    'organ',
    'site_type',
    'disease',
    'age',
    'sex',
    'category',
    'species'
]]  # Reorder the columns

cell_line_df

In [ ]:
cell_line_df.to_csv(f'input/filters/cell_lines_{cell_line_search_term}.csv', index=False)

### Filter melanoma cell lines

#### Melanoma brain metastasis cell lines

In [ ]:
df_mbm = cell_line_df[
    cell_line_df['organ'].str.contains('brain', case=False, na=False) &
    cell_line_df['site_type'].str.contains('metastatic', case=False, na=False)
] # These are all the melanoma that metastasizes to the brain

df_mbm = df_mbm.reset_index(drop=True)
df_mbm

In [ ]:
df_mbm.to_csv(f'input/filters/cell_lines_mbm.csv', index=False)

#### Uveal melanoma cell lines

In [ ]:
df_uveal = cell_line_df[
    cell_line_df['disease'].str.contains('uvea', case=False, na=False)
] #These are all the uveal melanoma, including the in situ and metastatic ones that can go to various locations (can be further specified)

df_uveal = df_uveal.reset_index(drop=True)
df_uveal

In [ ]:
df_mbm.to_csv(f'input/filters/cell_lines_um.csv', index=False)

## Obtain all astrocyte cell lines

In [ ]:
cell_line_search_term = 'astrocyte'
rows_size = 100

In [ ]:
all_cell_lines_data = obtain_all_cell_lines(cell_line_search_term, tax_id, rows_size)

In [ ]:
cell_line_df = cell_lines_df_clearing(all_cell_lines_data)
cell_line_df = cell_line_df[[
    'cell_line_id',
    'cell_line_name',
    'organ',
    'site_type',
    'disease',
    'age',
    'sex',
    'category',
    'species'
]]  # Reorder the columns

cell_line_df

In [ ]:
cell_line_df.to_csv(f'input/filters/cell_lines_{cell_line_search_term}.csv', index=False)